In [1]:
import polars as pl
import numpy as np

In [2]:
data = pl.DataFrame({
        'array_col': [[1] * 4000]  # Example array with 4000 elements
    })

data

array_col
list[i64]
"[1, 1, … 1]"


In [6]:
array_column = 'array_col'
chunk_size = 200

In [16]:
def reduce_array_columns(df, columns=None, chunk_size=200):
    # If no columns specified, use all columns
    if columns is None:
        columns = df.columns
    
    # Create expressions for all columns to be reduced
    expressions = [
        pl.col(col).map_elements(lambda x: 
            np.array(x)
            .reshape(-1, chunk_size)
            .sum(axis=1)
            .tolist(),
            return_dtype=pl.List(pl.Float64)
        ).alias(f"{col}_reduced")
        for col in columns
    ]
    
    return df.with_columns(expressions)

In [20]:
data = pl.DataFrame({
        'a': [[1.0] * 4000],
        'b': [[2.0] * 4000],
        'c': [[3.0] * 4000],
        'd': [[4.0] * 4000],
        'e': [[5.0] * 4000]
    })
    
# Specify columns to reduce
columns_to_reduce = ['a', 'b', 'c', 'd', 'e']

In [21]:
data

a,b,c,d,e
list[f64],list[f64],list[f64],list[f64],list[f64]
"[1.0, 1.0, … 1.0]","[2.0, 2.0, … 2.0]","[3.0, 3.0, … 3.0]","[4.0, 4.0, … 4.0]","[5.0, 5.0, … 5.0]"


In [22]:
result = reduce_array_columns(data, columns_to_reduce, chunk_size=200)

In [23]:
result

a,b,c,d,e,a_reduced,b_reduced,c_reduced,d_reduced,e_reduced
list[f64],list[f64],list[f64],list[f64],list[f64],list[f64],list[f64],list[f64],list[f64],list[f64]
"[1.0, 1.0, … 1.0]","[2.0, 2.0, … 2.0]","[3.0, 3.0, … 3.0]","[4.0, 4.0, … 4.0]","[5.0, 5.0, … 5.0]","[200.0, 200.0, … 200.0]","[400.0, 400.0, … 400.0]","[600.0, 600.0, … 600.0]","[800.0, 800.0, … 800.0]","[1000.0, 1000.0, … 1000.0]"


In [24]:
for col in columns_to_reduce:
    print(f"{col} - Original length: {len(data[col][0])}, "
    f"Reduced length: {len(result[f'{col}_reduced'][0])}")

a - Original length: 4000, Reduced length: 20
b - Original length: 4000, Reduced length: 20
c - Original length: 4000, Reduced length: 20
d - Original length: 4000, Reduced length: 20
e - Original length: 4000, Reduced length: 20
